# Omnipose Processing of Phalloidin Stained Images

Tries to segment phallodin stained actin fibers using omnipose.

## Environment Setup

In [ ]:
from omnipose.gpu import use_gpu
use_GPU = use_gpu()
use_GPU

import omnipose

# set up plotting defaults
omnipose.plot.setup()

# for plotting
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rcParams['figure.dpi'] = 300
plt.style.use('dark_background')
# %matplotlib inline

In [ ]:
from image_processing_tools.util.load_files import find_files_by_pattern
from image_processing_tools.image_class.image_container import ImageContainer
import numpy as np
import cv2
from pathlib import Path

def invert_image_colors(image: np.ndarray) -> np.ndarray:
    """
    Performs a color inversion on the input image.

    This function inverts the pixel values of an image. For an 8-bit image,
    a pixel with value `p` will become `255 - p`. For a 16-bit image,
    a pixel with value `p` will become `65535 - p`, and so on.
    It works for both grayscale and multi-channel (e.g., RGB) images.
    
    For logical matrices (boolean arrays), 0 (False) becomes 1 (True) and 
    1 (True) becomes 0 (False).

    Args:
        image (np.ndarray): The input image matrix. Can be grayscale (2D) or color (3D).

    Returns:
        np.ndarray: The color-inverted image.
    """
    if not isinstance(image, np.ndarray):
        raise TypeError("Input 'image' must be a NumPy array.")

    # Handle logical matrices (boolean arrays)
    if image.dtype == bool:
        return np.logical_not(image)

    # cv2.bitwise_not performs a bitwise NOT operation on each pixel.
    # For an 8-bit unsigned integer (uint8), this effectively inverts the color.
    # E.g., 0 (binary 00000000) becomes 255 (binary 11111111),
    # and 255 becomes 0.
    inverted_image = cv2.bitwise_not(image)

    return inverted_image

search_path = (
    "~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/CA_SSU_GAPDH_Infection_FOV1_1_CY5, CY3.5 NAR, FITC, DAPI, BF",
    "~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/CA_SSU_GAPDH_Infection_FOV2_1_CY5, CY3.5 NAR, FITC, DAPI",
    "~/A8/Data_Roan/250925_Cocu_Dyes_cFOS_CSF3_ECE1/Cocu_Cellbrite_cFOS_CSF3_ECE1_01_CY5, CY3.5 NAR, CY3, FITC, DAPI",
    "~/A8/Data_Roan/251114_MonoCa9_Phalloidin/CA9_Phalloidin_01_CY5, FITC, DAPI",
    # "~/A8/Data_Roan/251114_MonoCa9_Phalloidin/CA9_Phalloidin_01_CY5, FITC, DAPI, BF",
    "~/A8/Data_Roan/251205_MonoCa9_Phalloidin_Cellbrite_LowConf/Ca922_Mono_LowConf_Phal_Cellbrite_02_CY5, FITC, DAPI, BF",
    "~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_low_01_CY5, DAPI",
    "~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_low_01_DIC",
    )

file_pattern = (
    "*MAX*.tif",
    "*SHARPEST-normalized_variance*.tif",
    "*SHARPEST-manual*.tif",
    "SUBTRACT-direct*O5*.tif"
    )

config = {
    "preprocessing": {
        "resize_image": True,
        "max_dim": 1080,
        "outlier_percentile": 0.35,
        "quantization": "8bit"
    }
}

# Find the files. The 'files' variable will hold the list of Path objects.
files = find_files_by_pattern(search_path[6], file_pattern[2], verbose=True)

# cy5_container = ImageContainer(files[0],config)
# dapi_container = ImageContainer(files[2],config)
# dic_container = ImageContainer(files[1],config)

# Create output directory
base_input_dir = Path("~/A8/Data_Roan/").expanduser()
base_output_dir = Path("output/omnipose_phalloidin")

# Assuming 'files' is defined in your notebook environment as a list of Path objects
relative_dir = files[0].parent.relative_to(base_input_dir)
current_output_dir = base_output_dir / relative_dir
current_output_dir.mkdir(parents=True, exist_ok=True)

files = (files[4],)

## Iterate over all models

In [ ]:
# import matplotlib.pyplot as plt
# import time
# import os
# from pathlib import Path
# from cellpose_omni import models

# # Ensure environment variable is set
# os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# # Define models to iterate over
# model_names = [
#     'bact_phase_affinity',
#     'worm_high_res_omni',
#     'worm_bact_omni',
#     'cyto2_omni',
#     'bact_phase_cp',
#     'bact_fluor_cp',
#     'cyto2',
#     'cyto'
# ]

# # Define parameters
# params = {
#     'channels': None,
#     'rescale': None,
#     'mask_threshold': -2,
#     'flow_threshold': 0,
#     'transparency': True,
#     'omni': True,
#     'cluster': True,
#     'resample': True,
#     'verbose': False,
#     'tile': True,
#     'niter': None,
#     'augment': False,
#     'affinity_seg': False,
#     'diameter': 25
# }

# # Prepare the image once
# img = cy5_container.merge()

# # Initialize the figure for composite plotting
# # Rows = models, Cols = 3 (Image+Mask, Flow, Boundary)
# n_models = len(model_names)
# fig, axes = plt.subplots(n_models, 3, figsize=(15, 4 * n_models), constrained_layout=True)

# # Handle case of single model (axes would be 1D array)
# if n_models == 1:
#     axes = axes.reshape(1, -1)

# print(f"Starting evaluation of {n_models} models...")

# for i, model_name in enumerate(model_names):
#     print(f"[{i+1}/{n_models}] Running model: {model_name}")
    
#     # Initialize model
#     # Assuming 'use_GPU' is defined in your notebook environment
#     model = models.CellposeModel(gpu=use_GPU, model_type=model_name)
    
#     # Run segmentation
#     tic = time.time()
#     masks, flows, styles = model.eval(img, **params)
#     net_time = time.time() - tic
#     print(f'  -> Segmentation time: {net_time:.2f}s')
    
#     # Extract visualization components
#     # flows[0] is the RGB flow image
#     # flows[-1] is typically the boundary output in Omnipose
#     flow_rgb = flows[0]
#     boundary = invert_image_colors(flows[-1])
    
#     # --- Plotting ---
    
#     # 1. Image + Mask Overlay
#     ax_img = axes[i, 0]
#     ax_img.imshow(img, cmap='gray')
#     if masks.max() > 0:
#         # Overlay masks with transparency using a distinct colormap (e.g., nipy_spectral)
#         ax_img.imshow(masks, cmap='nipy_spectral', alpha=0.4, interpolation='nearest')
#     ax_img.set_title(f"{model_name}\nImage + Mask")
#     ax_img.axis('off')
    
#     # 2. Flow Field
#     ax_flow = axes[i, 1]
#     ax_flow.imshow(flow_rgb)
#     ax_flow.set_title("Flow Field")
#     ax_flow.axis('off')
    
#     # 3. Boundary
#     ax_bd = axes[i, 2]
#     ax_bd.imshow(boundary, cmap='gray')
#     ax_bd.set_title("Boundary")
#     ax_bd.axis('off')

# # --- Save Output ---
# base_input_dir = Path("~/A8/Data_Roan/").expanduser()
# base_output_dir = Path("output/omnipose_phalloidin")

# # Assuming 'files' is defined in your notebook environment as a list of Path objects
# relative_dir = files[0].parent.relative_to(base_input_dir)
# current_output_dir = base_output_dir / relative_dir
# current_output_dir.mkdir(parents=True, exist_ok=True)

# # Construct filename using the stem of the first file
# output_filename = current_output_dir / f"{files[0].stem}_omnipose_comparison.png"

# plt.savefig(output_filename, dpi=300)
# print(f"Saved comparison figure to: {output_filename}")
# plt.close(fig)
# # plt.show()

## Run one image with one model

In [ ]:
import os
from cellpose_omni import models

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# model_name = 'bact_phase_affinity'
# model_name = 'worm_high_res_omni'
# model_name = 'worm_bact_omni'
# model_name = 'cyto2_omni'
# model_name = 'bact_phase_cp'
model_name = 'bact_fluor_cp'
# model_name = 'cyto2'
# model_name = 'cyto'

model = models.CellposeModel(gpu=use_GPU, model_type=model_name)

### Save Segmentation Output

In [ ]:
import matplotlib
matplotlib.use('Agg') # Critical: prevents plotting to screen/RAM
import matplotlib.pyplot as plt
import gc
import time

# define parameters
params = {'channels':None, # always define this if using older models, e.g. [0,0] with bact_phase_omni
          'rescale': None, # upscale or downscale your images, None = no rescaling 
          'mask_threshold': -3, # erode or dilate masks with higher or lower values between -5 and 5 
          'flow_threshold': 0, # default is .4, but only needed if there are spurious masks to clean up; slows down output
          'transparency': True, # transparency in flow output
          'omni': True, # we can turn off Omnipose mask reconstruction, not advised 
          'cluster': True, # use DBSCAN clustering
          'resample': True, # whether or not to run dynamics on rescaled grid or original grid 
          'verbose': False, # turn on if you want to see more output 
          'tile': True, # average the outputs from flipped (augmented) images; slower, usually not needed 
          'niter': None, # default None lets Omnipose calculate # of Euler iterations (usually <20) but you can tune it for over/under segmentation 
          'augment': False, # Can optionally rotate the image and average network outputs, usually not needed 
          'affinity_seg': False, # new feature, stay tuned...
          'diameter': 140
         }

for f in files:

    img_c = ImageContainer(f,config)
    img_c.name

    img = img_c.merge()
    tic = time.time() 
    masks, flows, styles = model.eval(img, **params)

    net_time = time.time() - tic

    print('total segmentation time: {}s'.format(net_time))

    # Construct filename using the stem of the first file
    output_filename = current_output_dir / f"{Path(img_c.name).stem}_{model_name}.png"

    # Your original call was correct, but heavy. 
    # We wrap it to ensure memory is freed.
    try:
        fig, axes = plt.subplots(2, 2, figsize=(8, 8), constrained_layout=True)
        flow_rgb = flows[0]
        boundary = invert_image_colors(flows[-1])
        
        # 0. Image
        ax_img_only = axes[0,0]
        img_c.display(ax=ax_img_only)
        
        # 1. Image + Mask Overlay
        ax_img = axes[0,1]
        # ax_img.imshow(img, cmap='gray')
        img_c.display(ax=ax_img)
        if masks.max() > 0:
            # Overlay masks with transparency using a distinct colormap (e.g., nipy_spectral)
            ax_img.imshow(masks, cmap='nipy_spectral', alpha=0.4, interpolation='nearest')
        ax_img.set_title(f"{model_name}\nImage + Mask")
        ax_img.axis('off')
        
        # 2. Flow Field
        ax_flow = axes[1,0]
        im = ax_flow.imshow(flows[2], cmap='viridis')
        ax_flow.set_title("Flow Field")
        ax_flow.axis('off')
        
        cbar = fig.colorbar(im, ax=ax_flow, shrink=0.8)
        cbar.set_label('Cell Probability', rotation=270, labelpad=15)
        
        # 3. Boundary
        ax_bd = axes[1,1]
        ax_bd.imshow(boundary, cmap='gray')
        ax_bd.set_title("Boundary")
        ax_bd.axis('off')
            
        plt.savefig(output_filename, dpi=300, bbox_inches='tight')
        print(f"Saved: {output_filename}")

    finally:
        # Always close and clear, even if it crashes halfway
        if 'fig' in locals():
            plt.close(fig)
        gc.collect()


## Flow field vector display

Upon visual inspection, the bact_fluor_cp model is capturing lots of fiber structures. Next we try to display the directions of the flow field vectors on the image.

In [ ]:
import matplotlib
matplotlib.use('Agg') # Critical: prevents plotting to screen/RAM
import matplotlib.pyplot as plt
from matplotlib import colors
from cellpose_omni.plot import image_to_rgb, dx_to_circ
import gc

def quiver_plot(flow_rgb, dP, step=20, scale=1.0, width=0.005, alpha=0.8, 
                uniform_length=True, color_by_magnitude=True, cmap='viridis'):
    """
    Overlays arrows representing gradient direction and magnitude on the flow field image.

    Parameters
    ----------
    flow_rgb: 3D array (Ly, Lx, 3)
        The RGB flow field visualization (e.g. flows[0]).
    dP: 3D array (2, Ly, Lx)
        The flow field (dy, dx).
    step: int
        Spacing between arrows (in pixels).
    scale: float
        Scaling factor for arrow length.
    width: float
        Width of the arrow shaft.
    alpha: float
        Transparency of the arrows.
    uniform_length: bool
        If True, all arrows have the same length.
    color_by_magnitude: bool
        If True, colors arrows by gradient magnitude.
    cmap: str
        The name of the colormap to use if color_by_magnitude is True.
        
    Returns
    -------
    fig: matplotlib.figure.Figure
        The figure object.
    """
    import matplotlib.pyplot as plt
    from matplotlib import colors
    import numpy as np
    
    # dP is 2 x Ly x Lx
    Ly, Lx = dP.shape[-2:]
    
    # Grid
    y, x = np.mgrid[step//2:Ly:step, step//2:Lx:step]
    
    # Vectors
    # dP[0] is Y flow, dP[1] is X flow
    v = dP[0, step//2:Ly:step, step//2:Lx:step]
    u = dP[1, step//2:Ly:step, step//2:Lx:step]
    
    magnitude = np.sqrt(u**2 + v**2)
    mask = magnitude > 1e-5
    
    y, x = y[mask], x[mask]
    u, v = u[mask], v[mask]
    magnitude = magnitude[mask]
    
    if color_by_magnitude:
        norm = colors.Normalize(vmin=magnitude.min(), vmax=magnitude.max())
        cmap_obj = plt.get_cmap(cmap)
        arrow_colors = cmap_obj(norm(magnitude))
    else:
        arrow_colors = 'r'
        
    if uniform_length:
        u = u / magnitude
        v = v / magnitude
        # Scale for display
        length = step * scale * 0.8
        u *= length
        v *= length
        
    fig, ax = plt.subplots(figsize=(12,12))
    ax.imshow(flow_rgb)
    ax.quiver(x, y, u, v, color=arrow_colors, angles='xy', scale_units='xy', scale=1,
              width=width, pivot='mid', alpha=alpha)
    ax.axis('off')
    
    if color_by_magnitude:
        sm = plt.cm.ScalarMappable(cmap=plt.get_cmap(cmap), norm=norm)
        sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label('Flow Magnitude', color='black')
        cbar.ax.tick_params(labelcolor='black')
    
    return fig

fig = quiver_plot(dP = flows[1],
                  flow_rgb=flows[0])

# Construct filename using the stem of the first file
resized_dim = config.get('preprocessing').get('max_dim') if config.get('preprocessing').get('resize_image') else img.shape[0]
output_filename = current_output_dir / f"{Path(cy5_container.name).stem}_{resized_dim}_{model_name}_flow_vectors.png"

plt.savefig(output_filename, dpi=300, bbox_inches='tight')
plt.close(fig)
gc.collect()
print(f"Saved: {output_filename}")

## Visualize individual masks

In [ ]:
import imageio
import numpy as np
import cv2
from pathlib import Path

def save_masks_as_video(
    image: np.ndarray,
    masks: np.ndarray,
    output_path: Path,
    fps: int = 10,
    alpha: float = 0.5,
    batch_size: int = 10,
    bitrate_mbps: float = None
):
    """
    Creates a video showing masks being drawn in batches using imageio.

    Args:
        image (np.ndarray): The original image (2D or 3D RGB).
        masks (np.ndarray): The label mask image (2D).
        output_path (Path): Path to save the output video (e.g. .mp4).
        fps (int): Frames per second for the video.
        alpha (float): Transparency of the masks.
        batch_size (int): Number of masks to draw per frame.
        bitrate_mbps (float): Target bitrate in Mbps (e.g., 2.5 for 2.5 Mbps). 
                              If None, uses the encoder's default quality settings.
    """
    # Ensure output path is a string
    output_path_str = str(output_path)
    
    # Ensure image is RGB (H, W, 3) for imageio
    if image.ndim == 2:
        image_rgb = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    elif image.ndim == 3 and image.shape[2] == 3:
        # Assuming input is RGB (common in matplotlib/PIL workflows)
        image_rgb = image.copy()
    else:
        # Normalize and convert if dimensions are weird
        image_norm = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        if image_norm.ndim == 2:
             image_rgb = cv2.cvtColor(image_norm, cv2.COLOR_GRAY2RGB)
        elif image_norm.ndim == 3:
             image_rgb = cv2.cvtColor(image_norm, cv2.COLOR_RGB2BGR) # Assuming BGR input if weird

    # Normalize to uint8 if not already
    if image_rgb.dtype != np.uint8:
        image_rgb = cv2.normalize(image_rgb, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    unique_labels = np.unique(masks)
    unique_labels = unique_labels[unique_labels != 0]

    if len(unique_labels) == 0:
        print("No masks found to visualize.")
        return

    # Generate random colors (RGB)
    np.random.seed(42)
    colors = np.random.randint(0, 255, size=(unique_labels.max() + 1, 3), dtype=np.uint8)
    
    # Shuffle labels to draw random masks each time
    np.random.shuffle(unique_labels)

    # Configure imageio writer
    kwargs = {
        'fps': fps,
        'macro_block_size': None # Prevents resizing/cropping if dims aren't divisible by 16
    }
    
    if bitrate_mbps is not None:
        # imageio expects bitrate in bits per second (bps)
        kwargs['bitrate'] = int(bitrate_mbps * 1_000_000)

    try:
        writer = imageio.get_writer(output_path_str, **kwargs)
    except ImportError:
        print("Error: imageio-ffmpeg is required. Please install it with: pip install imageio[ffmpeg]")
        return

    canvas = image_rgb.copy()
    
    # Add initial frame
    writer.append_data(canvas)

    # Iterate in batches
    for i in range(0, len(unique_labels), batch_size):
        batch_labels = unique_labels[i : i + batch_size]
        
        for label in batch_labels:
            mask = masks == label
            color = colors[label]
            
            # Blend only in the mask region
            roi = canvas[mask].astype(np.float32)
            
            # Create colored overlay for the mask region
            overlay = np.full_like(roi, color)
            
            blended = roi * (1 - alpha) + overlay * alpha
            canvas[mask] = blended.astype(np.uint8)
        
        writer.append_data(canvas)

    writer.close()
    print(f"Saved video to {output_path}")

resized_dim = config.get('preprocessing').get('max_dim') if config.get('preprocessing').get('resize_image') else img.shape[0]
output_filename = current_output_dir / f"{Path(cy5_container.name).stem}_{resized_dim}_{model_name}_masks_overlay.mp4"
save_masks_as_video(
    image=img,
    masks=masks,
    output_path=output_filename,
    fps=10,
    alpha=0.5,
    batch_size=10,
    bitrate_mbps=6
)